# Introduction to Text-Fabric: A Microscope for Ancient Texts

*Why use code to read the NT?* To have Natural Language Processing tools undersanding the dictionary form of any word, its grammatical role, and an English translation.

That is what **Text-Fabric** does. It treats the ancient text as a natural language that can be processed and interpreted as an interconnected database. Every single word is a data point (called a **node**). Every node is tagged with extra information (called **features**).

In this notebook, we will use a digital version of the famous **Nestle 1904 (N1904) Greek New Testament**. We won't be doing any complex math or machine learning here—just learning how to ask the database to show us what it knows.

## About the Dataset and Tool

### The N1904 Greek New Testament

The dataset we use is the **Nestle 1904 Greek New Testament** (7th edition), prepared by the [Center for Biblical Languages and Cultures (CenterBLC)](https://centerblc.github.io/N1904/). It goes far beyond a plain text: every word is annotated with linguistic features (lemma, part of speech, morphology) and embedded in constituency syntax trees that allow structural analysis of sentences and clauses.

The full dataset is open-source and available on GitHub: [github.com/CenterBLC/N1904](https://github.com/CenterBLC/N1904) (MIT license, version 1.0.0, released July 2024).

### What is Text-Fabric?

[Text-Fabric](https://centerblc.github.io/N1904/tf.html) is a Python library developed by Dirk Roorda for working with ancient texts as richly annotated databases. It models a text as a set of **nodes** (words, phrases, clauses, verses…) connected by a network of **features** — the linguistic metadata attached to each node. This "warp and weft" structure lets you query, filter, and recombine the text in ways that are impossible with a plain file. Think of it as a microscope that lets you zoom from a single word all the way up to a whole book, while always keeping the linguistic annotations in view.

This is a fundamentally different approach from **[odyCy](https://centre-for-humanities-computing.github.io/odyCy/architecture.html)**, the spaCy-based pipeline used in other notebooks in this project. odyCy *computes* linguistic annotations on the fly — it runs raw Greek text through a sequence of NLP components (a fine-tuned BERT transformer, a morphologizer, a lemmatizer) to *predict* features such as lemma, part of speech, and dependency relations. Text-Fabric, by contrast, *stores* annotations that were curated by scholars ahead of time: you are querying a pre-built, human-verified dataset rather than asking a model to infer anything. The two approaches are complementary — odyCy is flexible and can process any Greek text, while Text-Fabric gives you authoritative, structured access to a specific critical edition.

## 1. Loading the Data

First, we need to load the Text-Fabric library and the N1904 database. 

Because this project comes with the data pre-installed locally, it will load almost instantly.

[Read more here](https://github.com/CenterBLC/N1904/blob/main/docs/tutorial/Load_the_Text-Fabric_dataset.ipynb) about that data being downloaded in your `~/text-fabric-data/github/CenterBLC/N1904/` local filesystem.

In [28]:
# Import the tool
from tf.app import use
import pandas as pd
from IPython.display import HTML

# Load the local N1904 Greek New Testament dataset
N1904 = use("CenterBLC/N1904", version="1.0.0", hoist=globals(), silent=True)

print("✅ Database loaded successfully!")

**Locating corpus resources ...**

Display is setup for viewtype [syntax-view](https://github.com/saulocantanhede/tfgreek2/blob/main/docs/syntax-view.md#start)

See [here](https://github.com/saulocantanhede/tfgreek2/blob/main/docs/viewtypes.md#start) for more information on viewtypes

✅ Database loaded successfully!


### The "Magic" Letters

When we loaded the database, it secretly gave us four tools to explore the text. Think of them as magic letters:

- **`F` (Features):** Tells you traits about a word (like its dictionary form or part of speech).
- **`T` (Text):** Reads the actual Greek words out loud for us.
- **`L` (Locality):** Helps us zoom in and out (e.g., from a word to the whole verse).
- **`N` (Nodes):** Helps us walk through the entire database word by word.

## 2. Displaying a Specific Verse

Let's use the **`T` (Text)** tool to look up a very famous verse: Mark 1:1.

In [29]:
# Find the specific section: Book of Mark, Chapter 1, Verse 1
verse_node = T.nodeFromSection(('Mark', 1, 1))

# Read the text for that verse
greek_text = T.text(verse_node)

print(f"Mark 1:1 \n{greek_text}")

Mark 1:1 
Ἀρχὴ τοῦ εὐαγγελίου Ἰησοῦ Χριστοῦ (Υἱοῦ Θεοῦ). 


## 3. Looking Under the Hood (Nodes & Features)

To understand *why* Text-Fabric is powerful, we need to look under the hood. 

Let's grab the first few words of Mark 1:1 and peek at their "Features" (the hidden data tags). We'll put this into a neat table so you can see exactly what the computer sees.

In [30]:
# Get all the individual word nodes inside Mark 1:1
words_in_verse = L.d(verse_node, otype='word')

data = []
# Look at just the first 5 words
for word_node in words_in_verse[:5]:
    data.append({
        "Word ID (Node)": word_node,
        "Greek Text": F.text.v(word_node),
        "Dictionary Form (Lemma)": F.lemma.v(word_node),
        "Part of Speech": F.sp.v(word_node),
        "English Gloss": F.gloss.v(word_node)
    })

# Display as a nice table
df = pd.DataFrame(data)
df

,Word ID (Node),Greek Text,Dictionary Form (Lemma),Part of Speech,English Gloss
0,18300,Ἀρχὴ,ἀρχή,subs,"beginning; ruler, authority"
1,18301,τοῦ,ὁ,art,the
2,18302,εὐαγγελίου,εὐαγγέλιον,subs,"gospel, the Good News"
3,18303,Ἰησοῦ,Ἰησοῦς,subs,"Jesus, Joshua"
4,18304,Χριστοῦ,Χριστός,subs,"the Anointed One, the Messiah, the Christ"


## 4. The Finale: Building an Interlinear Translation

Because we have access to the Greek text, the dictionary forms, the parts of speech, and the English glosses all separated neatly into data points, we can recombine them however we want.

Let's build a clean **Interlinear Display** for Mark 1:1-3, just like you would find in a printed study Bible.

We will present the text in a 3-line format:
1. **Greek Text**
2. **Dictionary Form (Part of Speech)** — e.g., `υἱός (subs)`
3. **Literal English Gloss**

In [31]:
def create_interlinear(book, start_chap, start_verse, end_verse):
    """
    Generates an HTML interlinear display for a range of verses.
    The middle line combines the lemma (dictionary form) and sp (part of speech).
    """
    html_output = "<div style='display: flex; flex-wrap: wrap; font-family: sans-serif;'>"
    
    for verse_num in range(start_verse, end_verse + 1):
        # Print verse number marker
        html_output += f"<div style='margin: 10px 15px 10px 0; font-weight: bold; font-size: 1.2em; color: #555;'>v{verse_num}</div>"
        
        verse_node = T.nodeFromSection((book, start_chap, verse_num))
        words = L.d(verse_node, otype='word')
        
        for w in words:
            text = F.text.v(w) or ""
            lemma = F.lemma.v(w) or ""
            sp = F.sp.v(w) or ""
            gloss = F.gloss.v(w) or ""
            
            # Format the middle line
            middle = ""
            if lemma and sp:
                middle = f"{lemma} ({sp})"
            elif lemma:
                middle = lemma
            elif sp:
                middle = f"({sp})"
                
            # Only show words that actually have text/glosses (skip pure punctuation nodes if they exist)
            if text.strip() or gloss.strip():
                html_output += f"""
                <div style='display: flex; flex-direction: column; align-items: center; margin: 10px 10px 20px 0; min-width: 50px;'>
                    <span style='font-size: 1.4em; margin-bottom: 2px;'>{text}</span>
                    <span style='font-size: 0.9em; color: #666; font-style: italic; margin-bottom: 2px;'>{middle}</span>
                    <span style='font-size: 0.9em; color: #d35400;'>{gloss}</span>
                </div>
                """
    
    html_output += "</div>"
    return HTML(html_output)

In [32]:
# Render the combined interlinear format for Mark 1:1-3
create_interlinear('Mark', 1, 1, 3)